# ENZAIme — Kaggle Training Notebook

Environment-Aware AI-Based Enzyme Suitability Prediction, Recommendation and Mutation Optimization System.

This notebook runs the **full ENZAIme data + training pipeline** (Section 33 of the project spec) end-to-end on Kaggle, using GPU when available:

1. Install dependencies
2. Locate the dataset (Kaggle input, or the repo's bundled demo data)
3. Load + validate enzyme data
4. Generate synthetic environmental scenarios
5. Generate ESM-2 embeddings (frozen encoder, cached to disk)
6. Build the model-ready dataset
7. Train Baseline 1 (pollutant-only) and Baseline 2 (rule-based reference)
8. Train the proposed model (ESM-2 + environment features -> small fusion network), if enough data + embeddings exist
9. Evaluate and compare
10. Save `model.pt`, `scaler.pkl`, `encoders.pkl`, `enzyme_metadata.csv`, `metrics.json` to `artifacts/`
11. Export everything the backend needs

**Scientific positioning:** this notebook does not claim to prove enzyme degradation. It trains a model to regress toward the project-defined transparent compatibility score (see `docs/scoring.md`), because no large matched enzyme-environment experimental suitability dataset exists yet. If training data is insufficient, the pipeline **gracefully stops after the baselines** and the application still works via the rule-based compatibility engine (`DEMO_MODE=true`) — this is a required fallback, not a bug.

> This notebook reuses the exact same `common/enzaime_core` package and `scripts/01-07` used by the local pipeline, so there is only one implementation of the scientific logic to keep in sync.


In [ ]:
# ============================================================
# 1. Install dependencies
# ============================================================
# Kaggle images usually already have torch + transformers + pandas installed.
# This cell is safe to re-run; it only installs what's missing.
import sys, subprocess

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

pip_install("transformers>=4.33", "biopython>=1.81", "scikit-learn>=1.3", "joblib>=1.3", "python-dotenv>=1.0")
print("Dependency installation complete.")


In [ ]:
# ============================================================
# 2. Locate the project (Kaggle dataset OR repo checkout)
# ============================================================
import os
from pathlib import Path

# When this notebook is attached to a Kaggle Dataset containing the ENZAIme
# repo (e.g. uploaded as "enzaime-mvp"), Kaggle mounts it under /kaggle/input/.
# We search a few likely locations and fall back to the current working
# directory (useful when running this notebook locally via `jupyter`).
CANDIDATES = [
    Path("/kaggle/input/enzaime-mvp/EnzAIme"),
    Path("/kaggle/input/enzaime-mvp"),
    Path("/kaggle/working/EnzAIme"),
    Path.cwd() / "EnzAIme",
    Path.cwd(),
]

REPO_ROOT = None
for c in CANDIDATES:
    if (c / "common" / "enzaime_core" / "config.py").exists():
        REPO_ROOT = c.resolve()
        break

if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the ENZAIme repo (looked for common/enzaime_core/config.py under "
        f"{[str(c) for c in CANDIDATES]}). If running on Kaggle, attach the ENZAIme project "
        "as a Dataset first (Add Data -> Upload -> select the EnzAIme_MVP folder/zip)."
    )

print(f"Using REPO_ROOT = {REPO_ROOT}")
sys.path.insert(0, str(REPO_ROOT / "common"))
os.chdir(REPO_ROOT)


In [ ]:
# ============================================================
# 3. GPU detection
# ============================================================
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device = {DEVICE}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))
else:
    print("No GPU detected — falling back to CPU. The smaller ESM-2 checkpoint "
          "(config.MODEL_NAME_FALLBACK) will be used automatically if the primary "
          "650M-parameter model is impractical on CPU/time budget.")

os.environ["DEVICE"] = DEVICE


In [ ]:
# ============================================================
# 4. Load + validate enzyme data (reuses scripts/01-03)
# ============================================================
import subprocess

for script in ["01_inspect_data.py", "02_clean_sequences.py", "03_build_master_dataset.py"]:
    print(f"\n=== Running scripts/{script} ===")
    result = subprocess.run([sys.executable, f"scripts/{script}"], capture_output=True, text=True)
    print(result.stdout[-3000:])
    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError(f"scripts/{script} failed")


In [ ]:
# ============================================================
# 5. Generate synthetic environmental scenarios (reuses scripts/04)
# ============================================================
result = subprocess.run([sys.executable, "scripts/04_generate_scenarios.py"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("scripts/04_generate_scenarios.py failed")


In [ ]:
# ============================================================
# 6. Generate ESM-2 embeddings (frozen encoder, cached to artifacts/embeddings)
# ============================================================
# MODEL_NAME defaults to facebook/esm2_t33_650M_UR50D (config.py). If this OOMs
# or is impractical given the Kaggle GPU/time budget, set the env var below to
# force the smaller fallback checkpoint before running this cell.
#
# os.environ["MODEL_NAME"] = "facebook/esm2_t6_8M_UR50D"  # uncomment to force small model

result = subprocess.run([sys.executable, "scripts/05_generate_embeddings.py"], capture_output=True, text=True, env=os.environ)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("scripts/05_generate_embeddings.py failed")


In [ ]:
# ============================================================
# 7-9. Build model-ready dataset, train baselines + proposed model
#       (reuses scripts/06 — includes enzyme-level CV/holdout splitting,
#        early stopping, and the REQUIRED graceful skip of the proposed
#        model if there isn't enough labeled+embedded data yet)
# ============================================================
result = subprocess.run([sys.executable, "scripts/06_train_model.py"], capture_output=True, text=True, env=os.environ)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("scripts/06_train_model.py failed")


In [ ]:
# ============================================================
# 10. Evaluate + compare baselines vs proposed model
# ============================================================
result = subprocess.run([sys.executable, "scripts/07_evaluate_model.py"], capture_output=True, text=True, env=os.environ)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("scripts/07_evaluate_model.py failed")


In [ ]:
# ============================================================
# 11. Inspect what was actually saved to artifacts/
# ============================================================
from enzaime_core import config
import json

print("artifacts/ contents:")
for p in sorted(config.ARTIFACTS_DIR.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(config.ARTIFACTS_DIR), f"({p.stat().st_size} bytes)")

metrics_path = config.ARTIFACTS_DIR / "metrics.json"
if metrics_path.exists():
    print("\nmetrics.json:")
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2, default=str))


In [ ]:
# ============================================================
# 12. Package artifacts for the backend
# ============================================================
# The FastAPI backend (backend/app/services/model_service.py) looks for:
#   artifacts/model.pt, artifacts/scaler.pkl, artifacts/encoders.pkl,
#   artifacts/config.json  -> enables AI Model mode (DEMO_MODE=false)
#
# If the proposed model was NOT trained (insufficient data on this run),
# these files simply won't exist, and the backend automatically serves the
# REQUIRED rule-based compatibility engine fallback instead — no code
# changes needed. Either way, copy the whole artifacts/ folder back into
# your local EnzAIme project's artifacts/ directory before running the API.

import shutil

output_dir = Path("/kaggle/working/enzaime_artifacts")
if output_dir.exists():
    shutil.rmtree(output_dir)
shutil.copytree(config.ARTIFACTS_DIR, output_dir)
print(f"Artifacts copied to {output_dir} for download from the Kaggle 'Output' tab.")
